## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-09-16-53-47 +0000,bbc,Trump questions midterm elections integrity af...,https://www.bbc.co.uk/news/articles/ck79p9350z...
1,2026-11-09-16-51-40 +0000,bbc,Why are hundreds of prisoners being released e...,https://www.bbc.co.uk/news/articles/cp3rxlyz6w...
2,2026-11-09-16-51-09 +0000,bbc,France accuses Iran of supermarket fridge hacking,https://www.bbc.co.uk/news/articles/cmg474vk6d...
3,2026-11-09-16-47-08 +0000,nyt,U.S. Commerce Dept. Inquiry Into Megaspeed Was...,https://www.nytimes.com/2026/11/09/business/ec...
4,2026-11-09-16-46-44 +0000,chitri,Illinois GOP chair says better messaging neede...,https://www.chicagotribune.com/2026/11/09/illi...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2678/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
0,trump,37
25,nvidia,28
58,ceo,16
60,huang,16
59,jensen,15


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
228,2026-11-09-00-30-00 +0000,wsj,President Trump said Sunday that he was joking...,https://www.wsj.com/politics/policy/trump-says...,151
46,2026-11-09-14-57-09 +0000,nypost,Nvidia stock plunges 5% after CEO Jensen Huang...,https://nypost.com/2026/11/09/business/nvidia-...,130
227,2026-11-09-00-31-25 +0000,wapo,Trump says he was ‘only kidding’ about offerin...,https://www.washingtonpost.com/politics/2026/1...,122
237,2026-11-08-23-56-01 +0000,cbc,Trump insists he was 'only kidding' about want...,https://www.cbc.ca/news/world/trump-dana-white...,118
126,2026-11-09-11-26-13 +0000,nypost,Trump says ‘I was only kidding’ with press sec...,https://nypost.com/2026/11/09/us-news/trump-sa...,114


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
228,151,2026-11-09-00-30-00 +0000,wsj,President Trump said Sunday that he was joking...,https://www.wsj.com/politics/policy/trump-says...
46,130,2026-11-09-14-57-09 +0000,nypost,Nvidia stock plunges 5% after CEO Jensen Huang...,https://nypost.com/2026/11/09/business/nvidia-...
211,66,2026-11-09-02-00-00 +0000,wsj,Voters fed up with high prices gave Democrats ...,https://www.wsj.com/politics/elections/democra...
118,46,2026-11-09-11-48-00 +0000,wsj,This week will pose two major tests for Presid...,https://www.wsj.com/politics/policy/nvidias-je...
58,43,2026-11-09-14-33-05 +0000,wapo,Stephen Miller starts job as chief of staff; G...,https://www.washingtonpost.com/politics/2026/1...
22,38,2026-11-09-15-56-49 +0000,nypost,Death toll from migrant boat found with bodies...,https://nypost.com/2026/11/09/world-news/death...
60,36,2026-11-09-14-28-31 +0000,nypost,Russell Brand sentenced to over 10 years in pr...,https://nypost.com/2026/11/09/world-news/russe...
80,33,2026-11-09-13-28-32 +0000,nypost,Republican Rep. French Hill won’t rule out imp...,https://nypost.com/2026/11/09/us-news/republic...
67,33,2026-11-09-14-05-21 +0000,nyt,"Suspect in Mass Shooting in Krakow, Poland, Di...",https://www.nytimes.com/2026/11/08/world/europ...
62,32,2026-11-09-14-24-21 +0000,wapo,U.S. gives Russia an early win even before Ukr...,https://www.washingtonpost.com/world/2026/11/0...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
